# DAIC-WOZ Multimodal Embeddings + Pooling Experiments

This notebook keeps chunk-level embeddings at 768 dimensions and makes pooling/reduction interchangeable downstream.

Saved per participant:
- WavLM Base+ layer 8: `N_audio × 768`
- MentalRoBERTa layer 12: `N_text × 768`
- `Participant_ID`, `Dataset`, `PHQ8_Binary`, `PHQ8_Score`
- chunk lengths and indices

Planned experiment grid:
1. Weighted mean, 768-D, no reduction
2. Attention, 768-D, no reduction
3. Weighted mean, 256-D, PCA
4. Attention, 256-D, PCA
5. Weighted mean, 256-D, learned linear projection
6. Attention, 256-D, learned linear projection

PCA, learned projection, attention pooling, scaling, and classifiers are fit/trained using training participants only.

In [1]:
# %pip install -U torch torchaudio transformers pandas numpy scikit-learn tqdm

from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, AutoFeatureExtractor, WavLMModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score

## 1. Configuration

Audio windows are created **in memory**; no 30-second WAV files are written.

For MentalRoBERTa, the usable number of content tokens is computed from the tokenizer rather than hard-coded. For a standard RoBERTa-style 512-position input, it is typically about 510 content tokens plus special tokens.

In [2]:
TRANSCRIPT_CSV = Path(
    "/Users/sonalprasad/Documents/Thesis/TFIDF/participant_documents.csv"
)

AUDIO_ROOT = Path(
    "/Volumes/LaCie/DAIC-WOZ/Processed Dataset"
)

OUTPUT_ROOT = Path(
    "/Users/sonalprasad/Documents/Thesis/precomputed_multimodal_embeddings"
)

TEXT_OUTPUT_DIR = OUTPUT_ROOT / "mentalroberta_layer12"
AUDIO_OUTPUT_DIR = OUTPUT_ROOT / "wavlm_base_plus_layer8"

TEXT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MENTAL_ROBERTA_MODEL = "mental/mental-roberta-base"
WAVLM_MODEL = "microsoft/wavlm-base-plus"

MENTAL_ROBERTA_LAYER = 12
WAVLM_LAYER = 8
RAW_HIDDEN_DIM = 768

AUDIO_CHUNK_SECONDS = 30.0
TARGET_SAMPLE_RATE = 16_000

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)
print("Output root:", OUTPUT_ROOT)

Device: mps
Output root: /Users/sonalprasad/Documents/Thesis/precomputed_multimodal_embeddings


In [3]:
participants_df = pd.read_csv(TRANSCRIPT_CSV)

required_columns = [
    "Participant_ID", "Dataset", "raw_participant_text",
    "PHQ8_Binary", "PHQ8_Score",
]

missing = [c for c in required_columns if c not in participants_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

participants_df["Participant_ID"] = (
    participants_df["Participant_ID"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

print("Participants:", len(participants_df))
display(participants_df[
    ["Participant_ID", "Dataset", "PHQ8_Binary", "PHQ8_Score"]
].head())

display(participants_df["Dataset"].value_counts(dropna=False))

Participants: 181


,Participant_ID,Dataset,PHQ8_Binary,PHQ8_Score
0,303,train,0,0
1,304,train,0,6
2,305,train,0,7
3,310,train,0,4
4,312,train,0,2


Dataset
train    103
test      46
dev       32
Name: count, dtype: int64

## 2. MentalRoBERTa extraction: save `N_text × 768`

`raw_participant_text → token chunks → MentalRoBERTa layer 12 → masked mean within each chunk → save every 768-D chunk vector`

The masked mean averages **content tokens only**, excluding padding and special tokens.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MENTAL_ROBERTA_MODEL)
text_model = AutoModel.from_pretrained(MENTAL_ROBERTA_MODEL)
text_model.to(DEVICE)
text_model.eval()

model_max_length = tokenizer.model_max_length
if model_max_length is None or model_max_length > 100_000:
    # Conservative fallback for models with a tokenizer sentinel.
    model_max_length = 512

num_special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
TEXT_CONTENT_TOKENS = model_max_length - num_special_tokens

print("Model max length:", model_max_length)
print("Special tokens:", num_special_tokens)
print("Content tokens/chunk:", TEXT_CONTENT_TOKENS)
print("Hidden size:", text_model.config.hidden_size)
print("Transformer layers:", text_model.config.num_hidden_layers)

assert text_model.config.hidden_size == RAW_HIDDEN_DIM
assert MENTAL_ROBERTA_LAYER <= text_model.config.num_hidden_layers

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/682 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at mental/mental-roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model max length: 512
Special tokens: 2
Content tokens/chunk: 510
Hidden size: 768
Transformer layers: 12


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [8]:
def chunk_token_ids(token_ids, chunk_size):
    return [
        token_ids[i:i + chunk_size]
        for i in range(0, len(token_ids), chunk_size)
    ]


def masked_content_mean(layer_hidden, special_tokens_mask, attention_mask):
    # layer_hidden: [1, seq_len, 768]
    # Keep real positions, but remove RoBERTa special tokens.
    content_mask = (
        attention_mask.bool()
        & (~special_tokens_mask.bool())
    )

    weights = content_mask.unsqueeze(-1).to(layer_hidden.dtype)
    denominator = weights.sum(dim=1).clamp_min(1.0)

    return (
        (layer_hidden * weights).sum(dim=1) / denominator
    ).squeeze(0)


@torch.inference_mode()
def extract_text_participant(text):

    if not isinstance(text, str) or not text.strip():
        return None

    # --------------------------------------------------
    # 1. Tokenize the FULL transcript without truncation
    #    and without adding special tokens.
    #
    #    verbose=False prevents the >512 warning.
    #    We are NOT feeding this full sequence to the model.
    # --------------------------------------------------

    full_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False,
        verbose=False,
    )

    if len(full_ids) == 0:
        return None

    print(
        f"Full transcript: {len(full_ids)} content tokens"
    )

    # --------------------------------------------------
    # 2. Split into model-safe chunks
    # --------------------------------------------------

    chunks = chunk_token_ids(
        full_ids,
        TEXT_CONTENT_TOKENS,
    )

    embeddings = []
    num_tokens = []
    token_starts = []
    token_ends = []

    cursor = 0

    # --------------------------------------------------
    # 3. Process one chunk at a time
    # --------------------------------------------------

    for chunk_index, ids in enumerate(chunks):

        # Add <s> and </s>
        prepared_ids = (
            tokenizer.build_inputs_with_special_tokens(
                ids
            )
        )

        input_ids = torch.tensor(
            [prepared_ids],
            dtype=torch.long,
            device=DEVICE,
        )

        attention_mask = torch.ones_like(
            input_ids
        )

        # --------------------------------------------------
        # Build the content mask ourselves.
        #
        # 1 = content token
        # 0 = special token
        #
        # Because we know exactly which token IDs we supplied,
        # this avoids get_special_tokens_mask().
        # --------------------------------------------------

        content_mask = torch.zeros_like(
            input_ids,
            dtype=torch.bool,
        )

        num_content_tokens = len(ids)

        # RoBERTa single sequence:
        #
        # <s> token1 token2 ... tokenN </s>
        #
        # Therefore content is positions 1 through N.
        content_mask[
            0,
            1 : 1 + num_content_tokens
        ] = True

        # Safety check
        assert (
            input_ids.shape[1]
            <= model_max_length
        ), (
            f"Chunk {chunk_index} has "
            f"{input_ids.shape[1]} tokens, "
            f"but model maximum is "
            f"{model_max_length}"
        )

        # --------------------------------------------------
        # 4. MentalRoBERTa forward pass
        # --------------------------------------------------

        outputs = text_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )

        # hidden_states[0] = embedding layer
        # hidden_states[1] = transformer layer 1
        # ...
        # hidden_states[12] = transformer layer 12

        layer_hidden = (
            outputs.hidden_states[
                MENTAL_ROBERTA_LAYER
            ]
        )

        # --------------------------------------------------
        # 5. Masked mean over CONTENT tokens only
        # --------------------------------------------------

        weights = (
            content_mask
            .unsqueeze(-1)
            .to(layer_hidden.dtype)
        )

        pooled = (
            (
                layer_hidden
                * weights
            )
            .sum(dim=1)
            /
            weights
            .sum(dim=1)
            .clamp_min(1.0)
        )

        pooled = pooled.squeeze(0)

        embeddings.append(
            pooled
            .cpu()
            .float()
            .numpy()
        )

        num_tokens.append(
            num_content_tokens
        )

        token_starts.append(
            cursor
        )

        token_ends.append(
            cursor
            + num_content_tokens
        )

        cursor += num_content_tokens

    # --------------------------------------------------
    # 6. Return N_chunks × 768
    # --------------------------------------------------

    return {
        "embeddings": np.stack(
            embeddings
        ).astype(np.float32),

        "num_tokens": np.asarray(
            num_tokens,
            dtype=np.int32,
        ),

        "chunk_indices": np.arange(
            len(chunks),
            dtype=np.int32,
        ),

        "token_start": np.asarray(
            token_starts,
            dtype=np.int32,
        ),

        "token_end": np.asarray(
            token_ends,
            dtype=np.int32,
        ),
    }

In [9]:
def save_text_embeddings(row, overwrite=False):
    pid = str(row["Participant_ID"])
    output_path = TEXT_OUTPUT_DIR / f"{pid}.npz"

    if output_path.exists() and not overwrite:
        return "skipped"

    result = extract_text_participant(row["raw_participant_text"])
    if result is None:
        return "empty_text"

    np.savez_compressed(
        output_path,
        participant_id=np.asarray(pid),
        dataset=np.asarray(str(row["Dataset"])),
        phq8_binary=np.asarray(row["PHQ8_Binary"]),
        phq8_score=np.asarray(row["PHQ8_Score"]),
        embeddings=result["embeddings"],
        num_tokens=result["num_tokens"],
        chunk_indices=result["chunk_indices"],
        token_start=result["token_start"],
        token_end=result["token_end"],
    )

    return "saved"


# Test one participant before extracting everyone.
example_row = participants_df.iloc[0]
example_text = extract_text_participant(example_row["raw_participant_text"])

print("Participant:", example_row["Participant_ID"])
print("Text matrix:", example_text["embeddings"].shape)
print("Tokens/chunk:", example_text["num_tokens"])

Full transcript: 2127 content tokens
Participant: 303
Text matrix: (5, 768)
Tokens/chunk: [510 510 510 510  87]


In [10]:
text_status = []
for _, row in tqdm(
    participants_df.iterrows(),
    total=len(participants_df),
    desc="MentalRoBERTa",
):
    text_status.append({
        "Participant_ID": row["Participant_ID"],
        "status": save_text_embeddings(row, overwrite=False),
    })

text_status_df = pd.DataFrame(text_status)
display(text_status_df["status"].value_counts())

MentalRoBERTa:   0%|          | 0/181 [00:00<?, ?it/s]

Full transcript: 2127 content tokens
Full transcript: 1111 content tokens
Full transcript: 3609 content tokens
Full transcript: 1368 content tokens
Full transcript: 1337 content tokens
Full transcript: 826 content tokens
Full transcript: 1531 content tokens
Full transcript: 742 content tokens
Full transcript: 816 content tokens
Full transcript: 844 content tokens
Full transcript: 794 content tokens
Full transcript: 852 content tokens
Full transcript: 1707 content tokens
Full transcript: 977 content tokens
Full transcript: 2008 content tokens
Full transcript: 667 content tokens
Full transcript: 891 content tokens
Full transcript: 2318 content tokens
Full transcript: 585 content tokens
Full transcript: 1609 content tokens
Full transcript: 1164 content tokens
Full transcript: 953 content tokens
Full transcript: 1122 content tokens
Full transcript: 696 content tokens
Full transcript: 1453 content tokens
Full transcript: 807 content tokens
Full transcript: 2292 content tokens
Full transcrip

status
saved    181
Name: count, dtype: int64

## 3. WavLM extraction: save `N_audio × 768`

The code searches each participant folder for `*_removed_interviewer_manually_cleaned.wav`.

Each recording is loaded, divided into non-overlapping 30-second tensors in memory, passed through WavLM Base+, and layer 8 is averaged across WavLM frames to produce one 768-D vector per audio chunk.

In [17]:
import torchaudio

wavlm_feature_extractor = AutoFeatureExtractor.from_pretrained(WAVLM_MODEL)
wavlm_model = WavLMModel.from_pretrained(WAVLM_MODEL)
wavlm_model.to(DEVICE)
wavlm_model.eval()

print("Hidden size:", wavlm_model.config.hidden_size)
print("Transformer layers:", wavlm_model.config.num_hidden_layers)

assert wavlm_model.config.hidden_size == RAW_HIDDEN_DIM
assert WAVLM_LAYER <= wavlm_model.config.num_hidden_layers


def find_clean_audio(participant_id):
    pid = str(participant_id)

    participant_dir = (
        AUDIO_ROOT
        / f"{pid}_P"
    )

    if not participant_dir.exists():
        return None

    matches = sorted(
        path
        for path in participant_dir.glob(
            "*_removed_interviewer_manually_cleaned.wav"
        )
        if not path.name.startswith("._")
    )

    if len(matches) == 0:
        return None

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple cleaned WAVs for participant "
            f"{pid}: {matches}"
        )

    return matches[0]


def load_audio_mono_16k(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sample_rate != TARGET_SAMPLE_RATE:
        waveform = torchaudio.functional.resample(
            waveform,
            orig_freq=sample_rate,
            new_freq=TARGET_SAMPLE_RATE,
        )
        sample_rate = TARGET_SAMPLE_RATE

    return waveform.squeeze(0), sample_rate


def waveform_chunks(waveform, sample_rate, chunk_seconds):
    chunk_samples = int(round(sample_rate * chunk_seconds))

    for start in range(0, waveform.numel(), chunk_samples):
        end = min(start + chunk_samples, waveform.numel())
        chunk = waveform[start:end]

        if chunk.numel() > 0:
            yield start, end, chunk


@torch.inference_mode()
def extract_audio_participant(audio_path):
    waveform, sample_rate = load_audio_mono_16k(audio_path)

    embeddings = []
    num_frames = []
    durations = []
    starts = []
    ends = []

    for start_sample, end_sample, chunk in waveform_chunks(
        waveform,
        sample_rate,
        AUDIO_CHUNK_SECONDS,
    ):
        inputs = wavlm_feature_extractor(
            chunk.numpy(),
            sampling_rate=sample_rate,
            return_tensors="pt",
            padding=False,
        )

        outputs = wavlm_model(
            input_values=inputs["input_values"].to(DEVICE),
            output_hidden_states=True,
            return_dict=True,
        )

        # hidden_states[0] = projected feature representation.
        # hidden_states[k] = output after transformer layer k.
        layer_hidden = outputs.hidden_states[WAVLM_LAYER]

        # One chunk -> one 768-D vector.
        pooled = layer_hidden.mean(dim=1).squeeze(0)

        embeddings.append(pooled.cpu().float().numpy())
        num_frames.append(layer_hidden.shape[1])
        durations.append(chunk.numel() / sample_rate)
        starts.append(start_sample / sample_rate)
        ends.append(end_sample / sample_rate)

    if not embeddings:
        return None

    return {
        "embeddings": np.stack(embeddings).astype(np.float32),
        "num_frames": np.asarray(num_frames, dtype=np.int32),
        "duration_seconds": np.asarray(durations, dtype=np.float32),
        "chunk_indices": np.arange(len(embeddings), dtype=np.int32),
        "start_seconds": np.asarray(starts, dtype=np.float32),
        "end_seconds": np.asarray(ends, dtype=np.float32),
    }

Hidden size: 768
Transformer layers: 12


In [14]:
def save_audio_embeddings(row, overwrite=False):
    pid = str(row["Participant_ID"])
    output_path = AUDIO_OUTPUT_DIR / f"{pid}.npz"

    if output_path.exists() and not overwrite:
        return "skipped"

    audio_path = find_clean_audio(pid)
    if audio_path is None:
        return "missing_audio"

    result = extract_audio_participant(audio_path)
    if result is None:
        return "empty_audio"

    np.savez_compressed(
        output_path,
        participant_id=np.asarray(pid),
        dataset=np.asarray(str(row["Dataset"])),
        phq8_binary=np.asarray(row["PHQ8_Binary"]),
        phq8_score=np.asarray(row["PHQ8_Score"]),
        source_audio_path=np.asarray(str(audio_path)),
        embeddings=result["embeddings"],
        num_frames=result["num_frames"],
        duration_seconds=result["duration_seconds"],
        chunk_indices=result["chunk_indices"],
        start_seconds=result["start_seconds"],
        end_seconds=result["end_seconds"],
    )

    return "saved"


for pid in participants_df["Participant_ID"].head(5):
    print(pid, "->", find_clean_audio(pid))

303 -> /Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_P/303_AUDIO_removed_interviewer_manually_cleaned.wav
304 -> /Volumes/LaCie/DAIC-WOZ/Processed Dataset/304_P/304_AUDIO_removed_interviewer_manually_cleaned.wav
305 -> /Volumes/LaCie/DAIC-WOZ/Processed Dataset/305_P/305_AUDIO_removed_interviewer_manually_cleaned.wav
310 -> /Volumes/LaCie/DAIC-WOZ/Processed Dataset/310_P/310_AUDIO_removed_interviewer_manually_cleaned.wav
312 -> /Volumes/LaCie/DAIC-WOZ/Processed Dataset/312_P/312_AUDIO_removed_interviewer_manually_cleaned.wav


In [15]:
# Test one participant before extracting all
example_audio_path = find_clean_audio(example_row["Participant_ID"])

if example_audio_path is not None:
    example_audio = extract_audio_participant(example_audio_path)
    print("Participant:", example_row["Participant_ID"])
    print("Audio:", example_audio_path)
    print("Audio matrix:", example_audio["embeddings"].shape)
    print("Chunk durations:", example_audio["duration_seconds"])
    print("WavLM frames/chunk:", example_audio["num_frames"])
else:
    print("No cleaned audio found for first participant.")

/Users/sonalprasad/Documents/Thesis/thesis-venv/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


Participant: 303
Audio: /Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_P/303_AUDIO_removed_interviewer_manually_cleaned.wav
Audio matrix: (23, 768)
Chunk durations: [30.    30.    30.    30.    30.    30.    30.    30.    30.    30.
 30.    30.    30.    30.    30.    30.    30.    30.    30.    30.
 30.    30.    11.354]
WavLM frames/chunk: [1499 1499 1499 1499 1499 1499 1499 1499 1499 1499 1499 1499 1499 1499
 1499 1499 1499 1499 1499 1499 1499 1499  567]


In [18]:
audio_status = []
for _, row in tqdm(
    participants_df.iterrows(),
    total=len(participants_df),
    desc="WavLM",
):
    audio_status.append({
        "Participant_ID": row["Participant_ID"],
        "status": save_audio_embeddings(row, overwrite=False),
    })

audio_status_df = pd.DataFrame(audio_status)
display(audio_status_df["status"].value_counts())

WavLM:   0%|          | 0/181 [00:00<?, ?it/s]

status
saved      93
skipped    88
Name: count, dtype: int64

## 4. Build a participant manifest

This verifies which participants have both modalities and records chunk counts.

In [19]:
manifest_rows = []

for _, row in participants_df.iterrows():
    pid = str(row["Participant_ID"])

    text_path = TEXT_OUTPUT_DIR / f"{pid}.npz"
    audio_path = AUDIO_OUTPUT_DIR / f"{pid}.npz"

    record = {
        "Participant_ID": pid,
        "Dataset": row["Dataset"],
        "PHQ8_Binary": row["PHQ8_Binary"],
        "PHQ8_Score": row["PHQ8_Score"],
        "text_embedding_path": str(text_path),
        "audio_embedding_path": str(audio_path),
        "has_text_embeddings": text_path.exists(),
        "has_audio_embeddings": audio_path.exists(),
        "n_text_chunks": np.nan,
        "n_audio_chunks": np.nan,
    }

    if text_path.exists():
        with np.load(text_path, allow_pickle=False) as d:
            record["n_text_chunks"] = d["embeddings"].shape[0]

    if audio_path.exists():
        with np.load(audio_path, allow_pickle=False) as d:
            record["n_audio_chunks"] = d["embeddings"].shape[0]

    manifest_rows.append(record)

manifest_df = pd.DataFrame(manifest_rows)

MANIFEST_PATH = OUTPUT_ROOT / "embedding_manifest.csv"
manifest_df.to_csv(MANIFEST_PATH, index=False)

display(manifest_df.head())
print("Manifest:", MANIFEST_PATH)

,Participant_ID,Dataset,PHQ8_Binary,PHQ8_Score,text_embedding_path,audio_embedding_path,has_text_embeddings,has_audio_embeddings,n_text_chunks,n_audio_chunks
0,303,train,0,0,/Users/sonalprasad/Documents/Thesis/precompute...,/Users/sonalprasad/Documents/Thesis/precompute...,True,True,5,23
1,304,train,0,6,/Users/sonalprasad/Documents/Thesis/precompute...,/Users/sonalprasad/Documents/Thesis/precompute...,True,True,3,17
2,305,train,0,7,/Users/sonalprasad/Documents/Thesis/precompute...,/Users/sonalprasad/Documents/Thesis/precompute...,True,True,8,6
3,310,train,0,4,/Users/sonalprasad/Documents/Thesis/precompute...,/Users/sonalprasad/Documents/Thesis/precompute...,True,True,3,21
4,312,train,0,2,/Users/sonalprasad/Documents/Thesis/precompute...,/Users/sonalprasad/Documents/Thesis/precompute...,True,True,3,19


Manifest: /Users/sonalprasad/Documents/Thesis/precomputed_multimodal_embeddings/embedding_manifest.csv


## 5. Canonical embedding loaders

These always load the original chunk matrices. Pooling and dimensionality reduction happen later.

In [20]:
def load_npz_embedding(path, length_key):
    with np.load(path, allow_pickle=False) as d:
        return {
            "participant_id": str(d["participant_id"].item()),
            "dataset": str(d["dataset"].item()),
            "phq8_binary": int(d["phq8_binary"].item()),
            "phq8_score": float(d["phq8_score"].item()),
            "embeddings": d["embeddings"].astype(np.float32),
            "lengths": d[length_key].astype(np.float32),
        }


def load_participant_modalities(participant_id):
    pid = str(participant_id)

    audio = load_npz_embedding(
        AUDIO_OUTPUT_DIR / f"{pid}.npz",
        "num_frames",
    )

    text = load_npz_embedding(
        TEXT_OUTPUT_DIR / f"{pid}.npz",
        "num_tokens",
    )

    return audio, text

## 6. Pooling modules

Weighted mean uses `num_frames` for audio and `num_tokens` for text. Attention learns its chunk weights from the training data.

In [21]:
def weighted_mean_pool_numpy(embeddings, lengths):
    embeddings = np.asarray(embeddings, dtype=np.float32)
    lengths = np.asarray(lengths, dtype=np.float32)

    weights = lengths / lengths.sum()
    return (embeddings * weights[:, None]).sum(axis=0)


class WeightedMeanPool(nn.Module):
    def forward(self, x, lengths):
        valid = lengths > 0
        weights = lengths.float() * valid.float()
        weights = weights / weights.sum(
            dim=1,
            keepdim=True,
        ).clamp_min(1e-8)

        pooled = (
            x * weights.unsqueeze(-1)
        ).sum(dim=1)

        return pooled, weights


class AttentionPool(nn.Module):
    def __init__(self, input_dim, attention_hidden_dim=128):
        super().__init__()

        self.score = nn.Sequential(
            nn.Linear(input_dim, attention_hidden_dim),
            nn.Tanh(),
            nn.Linear(attention_hidden_dim, 1),
        )

    def forward(self, x, lengths):
        valid = lengths > 0

        scores = self.score(x).squeeze(-1)
        scores = scores.masked_fill(
            ~valid,
            torch.finfo(scores.dtype).min,
        )

        weights = torch.softmax(scores, dim=1)

        pooled = (
            x * weights.unsqueeze(-1)
        ).sum(dim=1)

        return pooled, weights

## 7. Dimensionality reduction

The 768-D files are never overwritten.

- 768-D: identity
- 256-D PCA: fit separate audio/text PCA objects on training-participant chunks only
- 256-D learned projection: separate trainable `Linear(768, 256)` layers for audio/text

In [22]:
class IdentityProjection(nn.Module):
    def forward(self, x):
        return x


class LearnedLinearProjection(nn.Module):
    def __init__(self, input_dim=768, output_dim=256):
        super().__init__()
        self.projection = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.projection(x)


def fit_pca_from_training_participants(
    participant_ids,
    modality,
    n_components=256,
):
    matrices = []

    for pid in participant_ids:
        if modality == "audio":
            path = AUDIO_OUTPUT_DIR / f"{pid}.npz"
        elif modality == "text":
            path = TEXT_OUTPUT_DIR / f"{pid}.npz"
        else:
            raise ValueError("modality must be audio or text")

        with np.load(path, allow_pickle=False) as d:
            matrices.append(
                d["embeddings"].astype(np.float32)
            )

    X_train_chunks = np.concatenate(matrices, axis=0)

    pca = PCA(
        n_components=n_components,
        random_state=42,
    )
    pca.fit(X_train_chunks)

    return pca

## 8. Six experiment configurations

In [23]:
@dataclass(frozen=True)
class ExperimentConfig:
    name: str
    pooling: str
    dimension: int
    reduction: str


EXPERIMENTS = [
    ExperimentConfig("weighted_768", "weighted_mean", 768, "none"),
    ExperimentConfig("attention_768", "attention", 768, "none"),
    ExperimentConfig("weighted_256_pca", "weighted_mean", 256, "pca"),
    ExperimentConfig("attention_256_pca", "attention", 256, "pca"),
    ExperimentConfig("weighted_256_linear", "weighted_mean", 256, "linear"),
    ExperimentConfig("attention_256_linear", "attention", 256, "linear"),
]

display(pd.DataFrame([x.__dict__ for x in EXPERIMENTS]))

,name,pooling,dimension,reduction
0,weighted_768,weighted_mean,768,none
1,attention_768,attention,768,none
2,weighted_256_pca,weighted_mean,256,pca
3,attention_256_pca,attention,256,pca
4,weighted_256_linear,weighted_mean,256,linear
5,attention_256_linear,attention,256,linear


## 9. Variable-length participant dataset

Padding happens only across the chunk axis inside a mini-batch. Nothing padded is saved to disk.

In [24]:
class MultimodalChunkDataset(Dataset):
    def __init__(
        self,
        participant_ids,
        audio_pca=None,
        text_pca=None,
    ):
        self.participant_ids = [str(x) for x in participant_ids]
        self.audio_pca = audio_pca
        self.text_pca = text_pca

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        pid = self.participant_ids[index]
        audio, text = load_participant_modalities(pid)

        A = audio["embeddings"]
        T = text["embeddings"]

        if self.audio_pca is not None:
            A = self.audio_pca.transform(A).astype(np.float32)

        if self.text_pca is not None:
            T = self.text_pca.transform(T).astype(np.float32)

        if audio["phq8_binary"] != text["phq8_binary"]:
            raise ValueError(f"Label mismatch for {pid}")

        return {
            "participant_id": pid,
            "audio_embeddings": torch.from_numpy(A),
            "audio_lengths": torch.from_numpy(audio["lengths"]),
            "text_embeddings": torch.from_numpy(T),
            "text_lengths": torch.from_numpy(text["lengths"]),
            "label": torch.tensor(
                audio["phq8_binary"],
                dtype=torch.float32,
            ),
        }


def pad_chunk_batch(matrices, lengths):
    batch_size = len(matrices)
    max_chunks = max(x.shape[0] for x in matrices)
    dim = matrices[0].shape[1]

    X = torch.zeros(batch_size, max_chunks, dim)
    L = torch.zeros(batch_size, max_chunks)

    for i, (matrix, length) in enumerate(
        zip(matrices, lengths)
    ):
        n = matrix.shape[0]
        X[i, :n] = matrix
        L[i, :n] = length

    return X, L


def multimodal_collate(batch):
    audio_X, audio_L = pad_chunk_batch(
        [x["audio_embeddings"] for x in batch],
        [x["audio_lengths"] for x in batch],
    )

    text_X, text_L = pad_chunk_batch(
        [x["text_embeddings"] for x in batch],
        [x["text_lengths"] for x in batch],
    )

    return {
        "participant_id": [x["participant_id"] for x in batch],
        "audio_embeddings": audio_X,
        "audio_lengths": audio_L,
        "text_embeddings": text_X,
        "text_lengths": text_L,
        "labels": torch.stack(
            [x["label"] for x in batch]
        ),
    }

## 10. Trainable fusion model

For PCA experiments, PCA is applied by the dataset and the neural model receives 256-D chunks.

For learned-projection experiments, the model receives raw 768-D chunks and learns separate 768→256 projections for audio and text.

In [25]:
class MultimodalClassifier(nn.Module):
    def __init__(
        self,
        input_dim,
        output_dim,
        reduction,
        pooling,
        attention_hidden_dim=128,
        classifier_hidden_dim=128,
        dropout=0.2,
    ):
        super().__init__()

        if reduction == "none":
            if input_dim != output_dim:
                raise ValueError(
                    "Identity reduction requires equal dimensions."
                )
            self.audio_projection = IdentityProjection()
            self.text_projection = IdentityProjection()

        elif reduction == "linear":
            self.audio_projection = LearnedLinearProjection(
                input_dim, output_dim
            )
            self.text_projection = LearnedLinearProjection(
                input_dim, output_dim
            )

        else:
            raise ValueError(
                "Neural reduction must be none or linear. "
                "Apply PCA in the dataset."
            )

        if pooling == "weighted_mean":
            self.audio_pool = WeightedMeanPool()
            self.text_pool = WeightedMeanPool()
        elif pooling == "attention":
            self.audio_pool = AttentionPool(
                output_dim, attention_hidden_dim
            )
            self.text_pool = AttentionPool(
                output_dim, attention_hidden_dim
            )
        else:
            raise ValueError("Unknown pooling method.")

        self.classifier = nn.Sequential(
            nn.Linear(output_dim * 2, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim, 1),
        )

    def forward(
        self,
        audio_embeddings,
        audio_lengths,
        text_embeddings,
        text_lengths,
    ):
        A = self.audio_projection(audio_embeddings)
        T = self.text_projection(text_embeddings)

        audio_vector, audio_weights = self.audio_pool(
            A, audio_lengths
        )
        text_vector, text_weights = self.text_pool(
            T, text_lengths
        )

        fused = torch.cat(
            [audio_vector, text_vector],
            dim=-1,
        )

        logits = self.classifier(fused).squeeze(-1)

        return {
            "logits": logits,
            "fused": fused,
            "audio_weights": audio_weights,
            "text_weights": text_weights,
        }


def make_model(config):
    if config.dimension == 768:
        return MultimodalClassifier(
            input_dim=768,
            output_dim=768,
            reduction="none",
            pooling=config.pooling,
        ).to(DEVICE)

    if config.dimension == 256 and config.reduction == "linear":
        return MultimodalClassifier(
            input_dim=768,
            output_dim=256,
            reduction="linear",
            pooling=config.pooling,
        ).to(DEVICE)

    if config.dimension == 256 and config.reduction == "pca":
        return MultimodalClassifier(
            input_dim=256,
            output_dim=256,
            reduction="none",
            pooling=config.pooling,
        ).to(DEVICE)

    raise ValueError(f"Unsupported config: {config}")

## 11. Official DAIC-WOZ split IDs

This uses your existing `Dataset` column. If you later use participant-level cross-validation, replace these ID lists with fold-specific IDs; the embedding files stay unchanged.

In [26]:
available_df = manifest_df[
    manifest_df["has_text_embeddings"]
    & manifest_df["has_audio_embeddings"]
].copy()

train_ids = available_df.loc[
    available_df["Dataset"].eq("train"),
    "Participant_ID",
].astype(str).tolist()

dev_ids = available_df.loc[
    available_df["Dataset"].eq("dev"),
    "Participant_ID",
].astype(str).tolist()

test_ids = available_df.loc[
    available_df["Dataset"].eq("test"),
    "Participant_ID",
].astype(str).tolist()

print("Train:", len(train_ids))
print("Dev:", len(dev_ids))
print("Test:", len(test_ids))

Train: 103
Dev: 32
Test: 46


## 12. Prepare loaders for any experiment

In [39]:
def prepare_experiment_data(
    config,
    train_ids,
    dev_ids,
    test_ids,
    batch_size=8,
):
    audio_pca = None
    text_pca = None

    # ---------------------------------------------
    # Fit PCA ONLY on training participants
    # ---------------------------------------------
    if config.reduction == "pca":
        audio_pca = fit_pca_from_training_participants(
            train_ids,
            "audio",
            config.dimension,
        )

        text_pca = fit_pca_from_training_participants(
            train_ids,
            "text",
            config.dimension,
        )

    # ---------------------------------------------
    # Create datasets
    # ---------------------------------------------
    train_dataset = MultimodalChunkDataset(
        train_ids,
        audio_pca=audio_pca,
        text_pca=text_pca,
    )

    dev_dataset = MultimodalChunkDataset(
        dev_ids,
        audio_pca=audio_pca,
        text_pca=text_pca,
    )

    test_dataset = MultimodalChunkDataset(
        test_ids,
        audio_pca=audio_pca,
        text_pca=text_pca,
    )

    # ---------------------------------------------
    # Create data loaders
    # ---------------------------------------------
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=multimodal_collate,
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=multimodal_collate,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=multimodal_collate,
    )

    return {
        "train_loader": train_loader,
        "dev_loader": dev_loader,
        "test_loader": test_loader,
        "audio_pca": audio_pca,
        "text_pca": text_pca,
    }

## 13. Training/evaluation loop

Attention and learned projections are trained jointly with the binary classifier. Each experiment/fold gets a fresh model.

In [44]:
def binary_metrics(y_true, probabilities, threshold=0.5):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    metrics = {
        "accuracy": accuracy_score(
            y_true, predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
    }

    metrics["roc_auc"] = (
        roc_auc_score(y_true, probabilities)
        if len(np.unique(y_true)) == 2
        else np.nan
    )

    return metrics


def move_batch(batch):
    return {
        "participant_id": batch["participant_id"],
        "audio_embeddings": batch["audio_embeddings"].to(DEVICE),
        "audio_lengths": batch["audio_lengths"].to(DEVICE),
        "text_embeddings": batch["text_embeddings"].to(DEVICE),
        "text_lengths": batch["text_lengths"].to(DEVICE),
        "labels": batch["labels"].to(DEVICE),
    }


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    train=False,
):
    """
    Run one full epoch.

    If train=True:
        - gradients are enabled
        - optimizer updates model parameters

    If train=False:
        - no gradients
        - model is only evaluated

    Returns:
        loss
        y_true
        y_prob
        y_pred
        metrics
    """

    # ---------------------------------------------
    # Set train/evaluation mode
    # ---------------------------------------------

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_examples = 0

    labels_all = []
    probs_all = []

    # ---------------------------------------------
    # Loop through batches
    # ---------------------------------------------

    for batch in loader:

        batch = move_batch(batch)

        if train:
            optimizer.zero_grad()

        # Enable gradients only during training
        with torch.set_grad_enabled(train):

            out = model(
                batch["audio_embeddings"],
                batch["audio_lengths"],
                batch["text_embeddings"],
                batch["text_lengths"],
            )

            logits = out["logits"]

            labels = batch["labels"]

            loss = criterion(
                logits,
                labels,
            )

            if train:
                loss.backward()
                optimizer.step()

        # -----------------------------------------
        # Accumulate loss
        # -----------------------------------------

        batch_size_current = len(labels)

        total_loss += (
            loss.item()
            * batch_size_current
        )

        total_examples += (
            batch_size_current
        )

        # -----------------------------------------
        # Store predictions
        # -----------------------------------------

        probabilities = torch.sigmoid(
            logits
        )

        labels_all.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

        probs_all.extend(
            probabilities
            .detach()
            .cpu()
            .numpy()
            .tolist()
        )

    # ---------------------------------------------
    # Convert to numpy
    # ---------------------------------------------

    y_true = np.asarray(
        labels_all,
        dtype=int,
    )

    y_prob = np.asarray(
        probs_all,
        dtype=float,
    )

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    # ---------------------------------------------
    # Metrics
    # ---------------------------------------------

    metrics = binary_metrics(
        y_true,
        y_prob,
    )

    average_loss = (
        total_loss
        / total_examples
    )

    # ---------------------------------------------
    # Return everything
    # ---------------------------------------------

    return {
        "loss": average_loss,
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
        "metrics": metrics,
    }

In [55]:
import copy
import numpy as np
import pandas as pd
import torch


def train_experiment(
    config,
    train_ids,
    dev_ids,
    test_ids,
    max_epochs=50,
    patience=5,
    batch_size=8,
    learning_rate=1e-3,
    weight_decay=1e-4,
):
    """
    Train one multimodal experiment.

    train_ids:
        Used to learn model parameters.

    dev_ids:
        Used for early stopping and selecting
        the best model checkpoint.

    test_ids:
        Completely held out during training.
        Evaluated only after the best dev model
        has been restored.
    """

    # --------------------------------------------------
    # 1. Prepare datasets
    # --------------------------------------------------

    prepared = prepare_experiment_data(
        config=config,
        train_ids=train_ids,
        dev_ids=dev_ids,
        test_ids=test_ids,
        batch_size=batch_size,
    )

    train_loader = prepared["train_loader"]
    dev_loader = prepared["dev_loader"]
    test_loader = prepared["test_loader"]

    audio_pca = prepared.get("audio_pca")
    text_pca = prepared.get("text_pca")

    # --------------------------------------------------
    # 2. Create model
    # --------------------------------------------------

    if config.reduction == "pca":
        model_input_dim = config.dimension
        neural_reduction = "none"
    else:
        model_input_dim = 768
        neural_reduction = config.reduction
    
    
    model = MultimodalClassifier(
        input_dim=model_input_dim,
        output_dim=config.dimension,
        reduction=neural_reduction,
        pooling=config.pooling,
    ).to(DEVICE)

    # --------------------------------------------------
    # 3. Handle class imbalance
    # --------------------------------------------------

    train_labels = []

    for pid in train_ids:
        row = available_df[
            available_df["Participant_ID"].astype(str)
            == str(pid)
        ]

        if len(row) == 0:
            continue

        train_labels.append(
            int(row.iloc[0]["PHQ8_Binary"])
        )

    train_labels = np.asarray(
        train_labels,
        dtype=np.int64,
    )

    num_positive = (
        train_labels == 1
    ).sum()

    num_negative = (
        train_labels == 0
    ).sum()

    if num_positive == 0:
        raise ValueError(
            "Training set contains no positive examples."
        )

    pos_weight = (
        num_negative
        / num_positive
    )

    criterion = torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [pos_weight],
            dtype=torch.float32,
            device=DEVICE,
        )
    )
    #criterion = nn.BCEWithLogitsLoss()

    # --------------------------------------------------
    # 4. Optimizer
    # --------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    # --------------------------------------------------
    # 5. Early stopping state
    # --------------------------------------------------

    best_dev_balanced_accuracy = -float("inf")
    best_epoch = None
    best_model_state = None
    epochs_without_improvement = 0

    history = []

    # --------------------------------------------------
    # 6. Training loop
    # --------------------------------------------------

    for epoch in range(
        1,
        max_epochs + 1,
    ):

        # ------------------------------
        # Train
        # ------------------------------

        train_result = run_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            train=True,
        )

        # ------------------------------
        # Development
        # ------------------------------

        dev_result = run_epoch(
            model=model,
            loader=dev_loader,
            criterion=criterion,
            optimizer=None,
            train=False,
        )

        train_metrics = binary_metrics(
            train_result["y_true"],
            train_result["y_prob"],
        )

        dev_metrics = binary_metrics(
            dev_result["y_true"],
            dev_result["y_prob"],
        )

        epoch_row = {
            "epoch": epoch,

            "train_loss":
                train_result["loss"],

            "dev_loss":
                dev_result["loss"],
        }

        # Add train metrics
        for key, value in (
            train_metrics.items()
        ):
            epoch_row[
                f"train_{key}"
            ] = value

        # Add dev metrics
        for key, value in (
            dev_metrics.items()
        ):
            epoch_row[
                f"dev_{key}"
            ] = value

        history.append(
            epoch_row
        )

        # ------------------------------
        # Print concise progress
        # ------------------------------

        print(
            f"Epoch {epoch:02d} | "
            f"train loss: "
            f"{train_result['loss']:.4f} | "
            f"dev loss: "
            f"{dev_result['loss']:.4f} | "
            f"train bal acc: "
            f"{train_metrics['balanced_accuracy']:.4f} | "
            f"dev bal acc: "
            f"{dev_metrics['balanced_accuracy']:.4f}"
        )

        # --------------------------------------------------
        # 7. Check whether this is the best model
        # --------------------------------------------------

        current_dev_balanced_accuracy = (
            dev_metrics["balanced_accuracy"]
        )
        
        if (
            current_dev_balanced_accuracy
            > best_dev_balanced_accuracy
        ):
            best_dev_balanced_accuracy = (
                current_dev_balanced_accuracy
            )
        
            best_epoch = epoch
        
            best_model_state = copy.deepcopy(
                model.state_dict()
            )
        
            epochs_without_improvement = 0
        
        else:
            epochs_without_improvement += 1

        # --------------------------------------------------
        # 8. Early stopping
        # --------------------------------------------------

        if (
            epochs_without_improvement
            >= patience
        ):

            print(
                f"\nEarly stopping at "
                f"epoch {epoch}."
            )

            break

    # --------------------------------------------------
    # 9. Restore BEST dev model
    # --------------------------------------------------

    if best_model_state is None:
        raise RuntimeError(
            "No best model checkpoint was saved."
        )

    model.load_state_dict(
        best_model_state
    )

    print(
        f"Best epoch: {best_epoch}, "
        f"best dev balanced accuracy: "
        f"{best_dev_balanced_accuracy:.4f}"
    )

    # --------------------------------------------------
    # 10. Re-evaluate DEV using restored model
    # --------------------------------------------------

    best_dev_result = run_epoch(
        model=model,
        loader=dev_loader,
        criterion=criterion,
        optimizer=None,
        train=False,
    )

    best_dev_metrics = binary_metrics(
        best_dev_result["y_true"],
        best_dev_result["y_prob"],
    )

    # --------------------------------------------------
    # 11. Evaluate TEST exactly once
    # --------------------------------------------------

    test_result = run_epoch(
        model=model,
        loader=test_loader,
        criterion=criterion,
        optimizer=None,
        train=False,
    )

    test_metrics = binary_metrics(
        test_result["y_true"],
        test_result["y_prob"],
    )

    # Include losses
    best_dev_metrics["loss"] = (
        best_dev_result["loss"]
    )

    test_metrics["loss"] = (
        test_result["loss"]
    )

    # --------------------------------------------------
    # 12. Save participant-level test predictions
    # --------------------------------------------------

    test_predictions = pd.DataFrame({
        "true_label":
            test_result["y_true"],

        "probability":
            test_result["y_prob"],
    })

    test_predictions[
        "predicted_label"
    ] = (
        test_predictions[
            "probability"
        ] >= 0.5
    ).astype(int)

    # --------------------------------------------------
    # 13. Return everything
    # --------------------------------------------------

    return {
        "config": config,
        "model": model,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "best_dev_balanced_accuracy": (
            best_dev_balanced_accuracy
        ),
        "best_dev_metrics": best_dev_metrics,
        "test_metrics": test_metrics,
        "test_predictions": test_predictions,
        "audio_pca": audio_pca,
        "text_pca": text_pca,
    }

In [56]:
result = train_experiment(
    EXPERIMENTS[0],
    train_ids,
    dev_ids,
    test_ids,
    max_epochs=50,
    patience=5,
    batch_size=8,
)

Epoch 01 | train loss: 1.0352 | dev loss: 1.0591 | train bal acc: 0.4932 | dev bal acc: 0.4805
Epoch 02 | train loss: 1.0085 | dev loss: 1.0457 | train bal acc: 0.5270 | dev bal acc: 0.5000
Epoch 03 | train loss: 0.9777 | dev loss: 1.0739 | train bal acc: 0.6097 | dev bal acc: 0.5195
Epoch 04 | train loss: 0.9875 | dev loss: 1.0659 | train bal acc: 0.5881 | dev bal acc: 0.5801
Epoch 05 | train loss: 0.9495 | dev loss: 1.0849 | train bal acc: 0.6955 | dev bal acc: 0.5195
Epoch 06 | train loss: 0.9594 | dev loss: 1.0713 | train bal acc: 0.6678 | dev bal acc: 0.6515
Epoch 07 | train loss: 0.9478 | dev loss: 1.0606 | train bal acc: 0.6789 | dev bal acc: 0.5779
Epoch 08 | train loss: 0.9357 | dev loss: 1.0738 | train bal acc: 0.6293 | dev bal acc: 0.6515
Epoch 09 | train loss: 0.9199 | dev loss: 1.0794 | train bal acc: 0.7232 | dev bal acc: 0.6299
Epoch 10 | train loss: 0.9180 | dev loss: 1.0598 | train bal acc: 0.6999 | dev bal acc: 0.6039
Epoch 11 | train loss: 0.8986 | dev loss: 1.0852 |

In [58]:
display(
    pd.Series(
        result["test_metrics"],
        name="Weighted BCE Test Metrics",
    )
)

print(
    result["test_predictions"][
        "predicted_label"
    ].value_counts()
)

print(
    result["test_predictions"][
        "probability"
    ].describe()
)

accuracy             0.434783
balanced_accuracy    0.392857
f1                   0.235294
roc_auc              0.412946
loss                 1.044146
Name: Weighted BCE Test Metrics, dtype: float64

predicted_label
0    26
1    20
Name: count, dtype: int64
count    46.000000
mean      0.482768
std       0.039967
min       0.350007
25%       0.463106
50%       0.491968
75%       0.509010
max       0.535028
Name: probability, dtype: float64


In [59]:
y_true = result["test_predictions"]["true_label"].to_numpy()
y_pred = result["test_predictions"]["predicted_label"].to_numpy()
y_prob = result["test_predictions"]["probability"].to_numpy()

print("Test participants:", len(y_true))

print("\nTrue label counts:")
print(pd.Series(y_true).value_counts().sort_index())

print("\nPredicted label counts:")
print(pd.Series(y_pred).value_counts().sort_index())

print("\nProbability summary:")
print(pd.Series(y_prob).describe())

print("\nFirst 20:")
display(
    result["test_predictions"].head(20)
)

Test participants: 46

True label counts:
0    32
1    14
Name: count, dtype: int64

Predicted label counts:
0    26
1    20
Name: count, dtype: int64

Probability summary:
count    46.000000
mean      0.482768
std       0.039967
min       0.350007
25%       0.463106
50%       0.491968
75%       0.509010
max       0.535028
dtype: float64

First 20:


,true_label,probability,predicted_label
0,0,0.350007,0
1,0,0.505838,1
2,0,0.460470,0
3,1,0.417300,0
4,1,0.439842,0
5,1,0.438207,0
6,0,0.519920,1
7,0,0.491268,0
8,0,0.514692,1
9,1,0.503890,1


In [61]:
print("Best epoch:", result["best_epoch"])
print("Best dev balanced accuracy:", result["best_dev_balanced_accuracy"])

display(
    result["history"][
        [
            "epoch",
            "train_loss",
            "dev_loss",
            "train_accuracy",
            "dev_accuracy",
            "train_balanced_accuracy",
            "dev_balanced_accuracy",
            "train_f1",
            "dev_f1",
            "train_roc_auc",
            "dev_roc_auc",
        ]
    ]
)

Best epoch: 6
Best dev balanced accuracy: 0.6515151515151515


,epoch,train_loss,dev_loss,train_accuracy,dev_accuracy,train_balanced_accuracy,dev_balanced_accuracy,train_f1,dev_f1,train_roc_auc,dev_roc_auc
0,1,1.035245,1.059097,0.708738,0.37500,0.493243,0.480519,0.000000,0.473684,0.443150,0.610390
1,2,1.008486,1.045736,0.320388,0.34375,0.527027,0.500000,0.453125,0.511628,0.450606,0.614719
2,3,0.977665,1.073875,0.514563,0.62500,0.609739,0.519481,0.489796,0.250000,0.637931,0.567100
3,4,0.987525,1.065860,0.543689,0.56250,0.588071,0.580087,0.459770,0.500000,0.577819,0.580087
4,5,0.949455,1.084882,0.728155,0.62500,0.695480,0.519481,0.562500,0.250000,0.759553,0.571429
5,6,0.959364,1.071261,0.718447,0.65625,0.667754,0.651515,0.524590,0.560000,0.677074,0.580087
6,7,0.947843,1.060601,0.689320,0.53125,0.678938,0.577922,0.542857,0.516129,0.744641,0.580087
7,8,0.935690,1.073828,0.572816,0.65625,0.629310,0.651515,0.500000,0.560000,0.741379,0.575758
8,9,0.919873,1.079444,0.737864,0.65625,0.723206,0.629870,0.597015,0.521739,0.762815,0.575758
9,10,0.917997,1.059762,0.689320,0.59375,0.699907,0.603896,0.567568,0.518519,0.752097,0.597403


In [62]:
result_exp2 = train_experiment(
    EXPERIMENTS[1],
    train_ids,
    dev_ids,
    test_ids,
    max_epochs=50,
    patience=5,
    batch_size=8,
)

Epoch 01 | train loss: 1.0245 | dev loss: 1.0548 | train bal acc: 0.4949 | dev bal acc: 0.4567
Epoch 02 | train loss: 0.9822 | dev loss: 1.0982 | train bal acc: 0.5822 | dev bal acc: 0.5000
Epoch 03 | train loss: 0.9720 | dev loss: 1.0760 | train bal acc: 0.5531 | dev bal acc: 0.4697
Epoch 04 | train loss: 0.9515 | dev loss: 1.0743 | train bal acc: 0.6293 | dev bal acc: 0.5152
Epoch 05 | train loss: 0.9236 | dev loss: 1.0842 | train bal acc: 0.6535 | dev bal acc: 0.5606
Epoch 06 | train loss: 0.8907 | dev loss: 1.0919 | train bal acc: 0.6880 | dev bal acc: 0.5606
Epoch 07 | train loss: 0.8824 | dev loss: 1.1027 | train bal acc: 0.6277 | dev bal acc: 0.5606
Epoch 08 | train loss: 0.7992 | dev loss: 1.2315 | train bal acc: 0.7563 | dev bal acc: 0.5195
Epoch 09 | train loss: 0.8100 | dev loss: 1.1793 | train bal acc: 0.7050 | dev bal acc: 0.4913
Epoch 10 | train loss: 0.6952 | dev loss: 1.3225 | train bal acc: 0.8425 | dev bal acc: 0.5390

Early stopping at epoch 10.
Best epoch: 5, best d

In [63]:
display(
    pd.Series(
        result_exp2["test_metrics"],
        name="Experiment 2: Attention 768 Test Metrics",
    )
)

print(
    result_exp2["test_predictions"][
        "predicted_label"
    ].value_counts()
)

print(
    result_exp2["test_predictions"][
        "probability"
    ].describe()
)

accuracy             0.456522
balanced_accuracy    0.428571
f1                   0.285714
roc_auc              0.390625
loss                 1.079807
Name: Experiment 2: Attention 768 Test Metrics, dtype: float64

predicted_label
0    25
1    21
Name: count, dtype: int64
count    46.000000
mean      0.471508
std       0.072100
min       0.279801
25%       0.409596
50%       0.491122
75%       0.528042
max       0.589343
Name: probability, dtype: float64


In [64]:
y_true = result_exp2["test_predictions"]["true_label"].to_numpy()
y_pred = result_exp2["test_predictions"]["predicted_label"].to_numpy()
y_prob = result_exp2["test_predictions"]["probability"].to_numpy()

print("Test participants:", len(y_true))

print("\nTrue label counts:")
print(pd.Series(y_true).value_counts().sort_index())

print("\nPredicted label counts:")
print(pd.Series(y_pred).value_counts().sort_index())

print("\nProbability summary:")
print(pd.Series(y_prob).describe())

print("\nFirst 20:")
display(
    result_exp2["test_predictions"].head(20)
)

Test participants: 46

True label counts:
0    32
1    14
Name: count, dtype: int64

Predicted label counts:
0    25
1    21
Name: count, dtype: int64

Probability summary:
count    46.000000
mean      0.471508
std       0.072100
min       0.279801
25%       0.409596
50%       0.491122
75%       0.528042
max       0.589343
dtype: float64

First 20:


,true_label,probability,predicted_label
0,0,0.279801,0
1,0,0.515207,1
2,0,0.498146,0
3,1,0.429592,0
4,1,0.379557,0
5,1,0.397417,0
6,0,0.573246,1
7,0,0.445406,0
8,0,0.527806,1
9,1,0.491888,0


In [65]:
print("Best epoch:", result_exp2["best_epoch"])
print(
    "Best dev balanced accuracy:",
    result_exp2["best_dev_balanced_accuracy"],
)

display(
    result_exp2["history"][
        [
            "epoch",
            "train_loss",
            "dev_loss",
            "train_accuracy",
            "dev_accuracy",
            "train_balanced_accuracy",
            "dev_balanced_accuracy",
            "train_f1",
            "dev_f1",
            "train_roc_auc",
            "dev_roc_auc",
        ]
    ]
)

Best epoch: 5
Best dev balanced accuracy: 0.5606060606060606


,epoch,train_loss,dev_loss,train_accuracy,dev_accuracy,train_balanced_accuracy,dev_balanced_accuracy,train_f1,dev_f1,train_roc_auc,dev_roc_auc
0,1,1.024542,1.054838,0.349515,0.34375,0.494874,0.456710,0.417391,0.461538,0.434296,0.554113
1,2,0.982154,1.098234,0.640777,0.65625,0.582246,0.500000,0.412698,0.000000,0.601584,0.528139
2,3,0.971950,1.076008,0.689320,0.53125,0.553122,0.469697,0.304348,0.285714,0.638863,0.567100
3,4,0.951538,1.074331,0.572816,0.56250,0.629310,0.515152,0.500000,0.363636,0.684063,0.575758
4,5,0.923648,1.084214,0.728155,0.59375,0.653541,0.560606,0.500000,0.434783,0.736719,0.571429
5,6,0.890658,1.091914,0.747573,0.59375,0.688024,0.560606,0.551724,0.434783,0.763281,0.575758
6,7,0.882357,1.102679,0.495146,0.59375,0.627679,0.560606,0.509434,0.434783,0.789842,0.597403
7,8,0.799182,1.231487,0.815534,0.62500,0.756291,0.519481,0.654545,0.250000,0.856477,0.575758
8,9,0.810033,1.179294,0.621359,0.53125,0.705033,0.491342,0.571429,0.347826,0.837372,0.593074
9,10,0.695217,1.322549,0.864078,0.59375,0.842498,0.538961,0.766667,0.380952,0.917055,0.571429


In [66]:
result_exp3 = train_experiment(
    EXPERIMENTS[2],
    train_ids,
    dev_ids,
    test_ids,
    max_epochs=50,
    patience=5,
    batch_size=8,
)

Epoch 01 | train loss: 0.9946 | dev loss: 1.0689 | train bal acc: 0.5210 | dev bal acc: 0.5216
Epoch 02 | train loss: 0.9738 | dev loss: 1.0676 | train bal acc: 0.8036 | dev bal acc: 0.4481
Epoch 03 | train loss: 0.9516 | dev loss: 1.0662 | train bal acc: 0.8486 | dev bal acc: 0.6082
Epoch 04 | train loss: 0.9236 | dev loss: 1.0630 | train bal acc: 0.9452 | dev bal acc: 0.5606
Epoch 05 | train loss: 0.8852 | dev loss: 1.0570 | train bal acc: 0.9317 | dev bal acc: 0.6061
Epoch 06 | train loss: 0.8353 | dev loss: 1.0579 | train bal acc: 0.9145 | dev bal acc: 0.6061
Epoch 07 | train loss: 0.7775 | dev loss: 1.0554 | train bal acc: 0.9557 | dev bal acc: 0.6061
Epoch 08 | train loss: 0.7143 | dev loss: 1.0606 | train bal acc: 0.9557 | dev bal acc: 0.6061

Early stopping at epoch 8.
Best epoch: 3, best dev balanced accuracy: 0.6082


In [67]:
display(
    pd.Series(
        result_exp3["test_metrics"],
        name="Experiment 3: Weighted 256 PCA Test Metrics",
    )
)

print(
    result_exp3["test_predictions"][
        "predicted_label"
    ].value_counts()
)

print(
    result_exp3["test_predictions"][
        "probability"
    ].describe()
)

accuracy             0.565217
balanced_accuracy    0.406250
f1                   0.000000
roc_auc              0.375000
loss                 1.033752
Name: Experiment 3: Weighted 256 PCA Test Metrics, dtype: float64

predicted_label
0    40
1     6
Name: count, dtype: int64
count    46.000000
mean      0.485716
std       0.019881
min       0.410359
25%       0.483778
50%       0.492711
75%       0.496953
max       0.510693
Name: probability, dtype: float64


In [68]:
y_true = result_exp3["test_predictions"]["true_label"].to_numpy()
y_pred = result_exp3["test_predictions"]["predicted_label"].to_numpy()
y_prob = result_exp3["test_predictions"]["probability"].to_numpy()

print("Test participants:", len(y_true))

print("\nTrue label counts:")
print(pd.Series(y_true).value_counts().sort_index())

print("\nPredicted label counts:")
print(pd.Series(y_pred).value_counts().sort_index())

print("\nProbability summary:")
print(pd.Series(y_prob).describe())

print("\nFirst 20:")
display(
    result_exp3["test_predictions"].head(20)
)

Test participants: 46

True label counts:
0    32
1    14
Name: count, dtype: int64

Predicted label counts:
0    40
1     6
Name: count, dtype: int64

Probability summary:
count    46.000000
mean      0.485716
std       0.019881
min       0.410359
25%       0.483778
50%       0.492711
75%       0.496953
max       0.510693
dtype: float64

First 20:


,true_label,probability,predicted_label
0,0,0.410359,0
1,0,0.489388,0
2,0,0.459348,0
3,1,0.448776,0
4,1,0.457362,0
5,1,0.454319,0
6,0,0.510693,1
7,0,0.494145,0
8,0,0.492118,0
9,1,0.495781,0


In [69]:
print("Best epoch:", result_exp3["best_epoch"])
print(
    "Best dev balanced accuracy:",
    result_exp3["best_dev_balanced_accuracy"],
)

display(
    result_exp3["history"][
        [
            "epoch",
            "train_loss",
            "dev_loss",
            "train_accuracy",
            "dev_accuracy",
            "train_balanced_accuracy",
            "dev_balanced_accuracy",
            "train_f1",
            "dev_f1",
            "train_roc_auc",
            "dev_roc_auc",
        ]
    ]
)

Best epoch: 3
Best dev balanced accuracy: 0.6082251082251082


,epoch,train_loss,dev_loss,train_accuracy,dev_accuracy,train_balanced_accuracy,dev_balanced_accuracy,train_f1,dev_f1,train_roc_auc,dev_roc_auc
0,1,0.994634,1.068905,0.718447,0.65625,0.520969,0.521645,0.121212,0.153846,0.566170,0.450216
1,2,0.973837,1.067600,0.883495,0.53125,0.803588,0.448052,0.750000,0.210526,0.885368,0.515152
2,3,0.951569,1.066151,0.902913,0.65625,0.848555,0.608225,0.807692,0.476190,0.956664,0.567100
3,4,0.923628,1.063038,0.951456,0.59375,0.945247,0.560606,0.915254,0.434783,0.975769,0.562771
4,5,0.885153,1.056968,0.932039,0.62500,0.931733,0.606061,0.885246,0.500000,0.987884,0.575758
5,6,0.835321,1.057925,0.922330,0.62500,0.914492,0.606061,0.866667,0.500000,0.988350,0.567100
6,7,0.777547,1.055390,0.951456,0.62500,0.955732,0.606061,0.918033,0.500000,0.993476,0.575758
7,8,0.714302,1.060614,0.951456,0.62500,0.955732,0.606061,0.918033,0.500000,0.996272,0.575758


In [70]:
result_exp4 = train_experiment(
    EXPERIMENTS[3],
    train_ids,
    dev_ids,
    test_ids,
    max_epochs=50,
    patience=5,
    batch_size=8,
)

Epoch 01 | train loss: 0.9938 | dev loss: 1.0684 | train bal acc: 0.5210 | dev bal acc: 0.4762
Epoch 02 | train loss: 0.9729 | dev loss: 1.0627 | train bal acc: 0.8006 | dev bal acc: 0.5866
Epoch 03 | train loss: 0.9514 | dev loss: 1.0588 | train bal acc: 0.9212 | dev bal acc: 0.6082
Epoch 04 | train loss: 0.9245 | dev loss: 1.0540 | train bal acc: 0.9280 | dev bal acc: 0.6082
Epoch 05 | train loss: 0.8775 | dev loss: 1.0501 | train bal acc: 0.9452 | dev bal acc: 0.5649
Epoch 06 | train loss: 0.8171 | dev loss: 1.0519 | train bal acc: 0.9692 | dev bal acc: 0.5649
Epoch 07 | train loss: 0.7248 | dev loss: 1.0927 | train bal acc: 0.9865 | dev bal acc: 0.5887
Epoch 08 | train loss: 0.6162 | dev loss: 1.2431 | train bal acc: 0.9760 | dev bal acc: 0.5887

Early stopping at epoch 8.
Best epoch: 3, best dev balanced accuracy: 0.6082


In [71]:
display(
    pd.Series(
        result_exp4["test_metrics"],
        name="Experiment 4: Attention 256 PCA Test Metrics",
    )
)

print(
    result_exp4["test_predictions"][
        "predicted_label"
    ].value_counts()
)

print(
    result_exp4["test_predictions"][
        "probability"
    ].describe()
)

accuracy             0.565217
balanced_accuracy    0.446429
f1                   0.166667
roc_auc              0.433036
loss                 1.028453
Name: Experiment 4: Attention 256 PCA Test Metrics, dtype: float64

predicted_label
0    36
1    10
Name: count, dtype: int64
count    46.000000
mean      0.486559
std       0.016354
min       0.426736
25%       0.478578
50%       0.489972
75%       0.499026
max       0.507203
Name: probability, dtype: float64


In [72]:
y_true = result_exp4["test_predictions"]["true_label"].to_numpy()
y_pred = result_exp4["test_predictions"]["predicted_label"].to_numpy()
y_prob = result_exp4["test_predictions"]["probability"].to_numpy()

print("Test participants:", len(y_true))

print("\nTrue label counts:")
print(pd.Series(y_true).value_counts().sort_index())

print("\nPredicted label counts:")
print(pd.Series(y_pred).value_counts().sort_index())

print("\nProbability summary:")
print(pd.Series(y_prob).describe())

print("\nFirst 20:")
display(
    result_exp4["test_predictions"].head(20)
)

Test participants: 46

True label counts:
0    32
1    14
Name: count, dtype: int64

Predicted label counts:
0    36
1    10
Name: count, dtype: int64

Probability summary:
count    46.000000
mean      0.486559
std       0.016354
min       0.426736
25%       0.478578
50%       0.489972
75%       0.499026
max       0.507203
dtype: float64

First 20:


,true_label,probability,predicted_label
0,0,0.426736,0
1,0,0.491329,0
2,0,0.472523,0
3,1,0.450797,0
4,1,0.468705,0
5,1,0.466712,0
6,0,0.506688,1
7,0,0.482001,0
8,0,0.481781,0
9,1,0.499655,0


In [73]:
print("Best epoch:", result_exp4["best_epoch"])
print(
    "Best dev balanced accuracy:",
    result_exp4["best_dev_balanced_accuracy"],
)

display(
    result_exp4["history"][
        [
            "epoch",
            "train_loss",
            "dev_loss",
            "train_accuracy",
            "dev_accuracy",
            "train_balanced_accuracy",
            "dev_balanced_accuracy",
            "train_f1",
            "dev_f1",
            "train_roc_auc",
            "dev_roc_auc",
        ]
    ]
)

Best epoch: 3
Best dev balanced accuracy: 0.6082251082251082


,epoch,train_loss,dev_loss,train_accuracy,dev_accuracy,train_balanced_accuracy,dev_balanced_accuracy,train_f1,dev_f1,train_roc_auc,dev_roc_auc
0,1,0.993830,1.068422,0.718447,0.62500,0.520969,0.476190,0.121212,0.000000,0.584809,0.536797
1,2,0.972874,1.062735,0.864078,0.65625,0.800559,0.586580,0.730769,0.421053,0.892358,0.584416
2,3,0.951428,1.058849,0.932039,0.65625,0.921249,0.608225,0.881356,0.476190,0.959459,0.597403
3,4,0.924450,1.054033,0.941748,0.65625,0.928006,0.608225,0.896552,0.476190,0.978565,0.619048
4,5,0.877484,1.050079,0.951456,0.65625,0.945247,0.564935,0.915254,0.352941,0.989748,0.658009
5,6,0.817072,1.051907,0.970874,0.65625,0.969245,0.564935,0.949153,0.352941,0.998602,0.627706
6,7,0.724826,1.092668,0.980583,0.68750,0.986486,0.588745,0.966667,0.375000,0.999068,0.597403
7,8,0.616210,1.243084,0.980583,0.68750,0.976002,0.588745,0.965517,0.375000,0.999534,0.575758


In [74]:
result_exp5 = train_experiment(
    EXPERIMENTS[4],
    train_ids,
    dev_ids,
    test_ids,
    max_epochs=50,
    patience=5,
    batch_size=8,
)

Epoch 01 | train loss: 1.0086 | dev loss: 1.0880 | train bal acc: 0.4583 | dev bal acc: 0.5000
Epoch 02 | train loss: 1.0014 | dev loss: 1.0971 | train bal acc: 0.4674 | dev bal acc: 0.5000
Epoch 03 | train loss: 1.0043 | dev loss: 1.0611 | train bal acc: 0.5284 | dev bal acc: 0.5779
Epoch 04 | train loss: 1.0174 | dev loss: 1.0654 | train bal acc: 0.4786 | dev bal acc: 0.6017
Epoch 05 | train loss: 0.9743 | dev loss: 1.0614 | train bal acc: 0.6375 | dev bal acc: 0.5779
Epoch 06 | train loss: 0.9886 | dev loss: 1.0703 | train bal acc: 0.5501 | dev bal acc: 0.5801
Epoch 07 | train loss: 0.9370 | dev loss: 1.0900 | train bal acc: 0.5948 | dev bal acc: 0.6061
Epoch 08 | train loss: 0.9058 | dev loss: 1.1018 | train bal acc: 0.6549 | dev bal acc: 0.6061
Epoch 09 | train loss: 0.8981 | dev loss: 1.0518 | train bal acc: 0.6445 | dev bal acc: 0.5022
Epoch 10 | train loss: 0.8506 | dev loss: 1.0405 | train bal acc: 0.6999 | dev bal acc: 0.5281
Epoch 11 | train loss: 0.8254 | dev loss: 1.0491 |

In [75]:
display(
    pd.Series(
        result_exp5["test_metrics"],
        name="Experiment 5: Weighted 256 Linear Test Metrics",
    )
)

print(
    result_exp5["test_predictions"][
        "predicted_label"
    ].value_counts()
)

print(
    result_exp5["test_predictions"][
        "probability"
    ].describe()
)

accuracy             0.456522
balanced_accuracy    0.388393
f1                   0.193548
roc_auc              0.390625
loss                 1.084378
Name: Experiment 5: Weighted 256 Linear Test Metrics, dtype: float64

predicted_label
0    29
1    17
Name: count, dtype: int64
count    46.000000
mean      0.468838
std       0.071318
min       0.265264
25%       0.430018
50%       0.483577
75%       0.514888
max       0.564228
Name: probability, dtype: float64


In [76]:
y_true = result_exp5["test_predictions"]["true_label"].to_numpy()
y_pred = result_exp5["test_predictions"]["predicted_label"].to_numpy()
y_prob = result_exp5["test_predictions"]["probability"].to_numpy()

print("Test participants:", len(y_true))

print("\nTrue label counts:")
print(pd.Series(y_true).value_counts().sort_index())

print("\nPredicted label counts:")
print(pd.Series(y_pred).value_counts().sort_index())

print("\nProbability summary:")
print(pd.Series(y_prob).describe())

print("\nFirst 20:")
display(
    result_exp5["test_predictions"].head(20)
)

Test participants: 46

True label counts:
0    32
1    14
Name: count, dtype: int64

Predicted label counts:
0    29
1    17
Name: count, dtype: int64

Probability summary:
count    46.000000
mean      0.468838
std       0.071318
min       0.265264
25%       0.430018
50%       0.483577
75%       0.514888
max       0.564228
dtype: float64

First 20:


,true_label,probability,predicted_label
0,0,0.265264,0
1,0,0.492259,0
2,0,0.429254,0
3,1,0.330179,0
4,1,0.381275,0
5,1,0.379468,0
6,0,0.541858,1
7,0,0.476976,0
8,0,0.530404,1
9,1,0.492843,0


In [77]:
print("Best epoch:", result_exp5["best_epoch"])
print(
    "Best dev balanced accuracy:",
    result_exp5["best_dev_balanced_accuracy"],
)

display(
    result_exp5["history"][
        [
            "epoch",
            "train_loss",
            "dev_loss",
            "train_accuracy",
            "dev_accuracy",
            "train_balanced_accuracy",
            "dev_balanced_accuracy",
            "train_f1",
            "dev_f1",
            "train_roc_auc",
            "dev_roc_auc",
        ]
    ]
)

Best epoch: 7
Best dev balanced accuracy: 0.606060606060606


,epoch,train_loss,dev_loss,train_accuracy,dev_accuracy,train_balanced_accuracy,dev_balanced_accuracy,train_f1,dev_f1,train_roc_auc,dev_roc_auc
0,1,1.008569,1.087961,0.417476,0.65625,0.458295,0.500000,0.347826,0.000000,0.449674,0.484848
1,2,1.001447,1.097066,0.475728,0.65625,0.467381,0.500000,0.325000,0.000000,0.473905,0.519481
2,3,1.004345,1.061139,0.699029,0.53125,0.528425,0.577922,0.205128,0.516129,0.545666,0.567100
3,4,1.017420,1.065382,0.446602,0.56250,0.478565,0.601732,0.359551,0.533333,0.457130,0.567100
4,5,0.974329,1.061442,0.524272,0.53125,0.637465,0.577922,0.514851,0.516129,0.605778,0.571429
5,6,0.988568,1.070276,0.669903,0.56250,0.550093,0.580087,0.320000,0.500000,0.576421,0.575758
6,7,0.937044,1.089966,0.553398,0.62500,0.594828,0.606061,0.465116,0.500000,0.726002,0.575758
7,8,0.905789,1.101840,0.669903,0.62500,0.654939,0.606061,0.514286,0.500000,0.727866,0.580087
8,9,0.898105,1.051811,0.669903,0.37500,0.644455,0.502165,0.500000,0.500000,0.695247,0.601732
9,10,0.850622,1.040512,0.689320,0.43750,0.699907,0.528139,0.567568,0.500000,0.748835,0.606061


In [78]:
result_exp6 = train_experiment(
    EXPERIMENTS[5],
    train_ids,
    dev_ids,
    test_ids,
    max_epochs=50,
    patience=5,
    batch_size=8,
)

Epoch 01 | train loss: 0.9899 | dev loss: 1.1161 | train bal acc: 0.5384 | dev bal acc: 0.5000
Epoch 02 | train loss: 1.0024 | dev loss: 1.0537 | train bal acc: 0.5801 | dev bal acc: 0.4329
Epoch 03 | train loss: 1.0021 | dev loss: 1.0552 | train bal acc: 0.5417 | dev bal acc: 0.5758
Epoch 04 | train loss: 1.0094 | dev loss: 1.0862 | train bal acc: 0.5503 | dev bal acc: 0.4762
Epoch 05 | train loss: 0.9778 | dev loss: 1.1103 | train bal acc: 0.5345 | dev bal acc: 0.5000
Epoch 06 | train loss: 1.0027 | dev loss: 1.0419 | train bal acc: 0.5354 | dev bal acc: 0.6472
Epoch 07 | train loss: 1.0056 | dev loss: 1.0528 | train bal acc: 0.5075 | dev bal acc: 0.6277
Epoch 08 | train loss: 0.9029 | dev loss: 1.0045 | train bal acc: 0.6270 | dev bal acc: 0.5758
Epoch 09 | train loss: 0.7901 | dev loss: 1.3901 | train bal acc: 0.7307 | dev bal acc: 0.5671
Epoch 10 | train loss: 0.6452 | dev loss: 1.3830 | train bal acc: 0.8057 | dev bal acc: 0.5887
Epoch 11 | train loss: 0.6784 | dev loss: 1.2708 |

In [79]:
display(
    pd.Series(
        result_exp6["test_metrics"],
        name="Experiment 6: Attention 256 Linear Test Metrics",
    )
)

print(
    result_exp6["test_predictions"][
        "predicted_label"
    ].value_counts()
)

print(
    result_exp6["test_predictions"][
        "probability"
    ].describe()
)

accuracy             0.391304
balanced_accuracy    0.421875
f1                   0.333333
roc_auc              0.511161
loss                 1.026211
Name: Experiment 6: Attention 256 Linear Test Metrics, dtype: float64

predicted_label
1    28
0    18
Name: count, dtype: int64
count    46.000000
mean      0.504628
std       0.040173
min       0.378766
25%       0.478770
50%       0.515490
75%       0.532154
max       0.579006
Name: probability, dtype: float64


In [80]:
y_true = result_exp6["test_predictions"]["true_label"].to_numpy()
y_pred = result_exp6["test_predictions"]["predicted_label"].to_numpy()
y_prob = result_exp6["test_predictions"]["probability"].to_numpy()

print("Test participants:", len(y_true))

print("\nTrue label counts:")
print(pd.Series(y_true).value_counts().sort_index())

print("\nPredicted label counts:")
print(pd.Series(y_pred).value_counts().sort_index())

print("\nProbability summary:")
print(pd.Series(y_prob).describe())

print("\nFirst 20:")
display(
    result_exp6["test_predictions"].head(20)
)

Test participants: 46

True label counts:
0    32
1    14
Name: count, dtype: int64

Predicted label counts:
0    18
1    28
Name: count, dtype: int64

Probability summary:
count    46.000000
mean      0.504628
std       0.040173
min       0.378766
25%       0.478770
50%       0.515490
75%       0.532154
max       0.579006
dtype: float64

First 20:


,true_label,probability,predicted_label
0,0,0.378766,0
1,0,0.528148,1
2,0,0.470376,0
3,1,0.426938,0
4,1,0.480557,0
5,1,0.459774,0
6,0,0.544380,1
7,0,0.518665,1
8,0,0.537838,1
9,1,0.552746,1


In [81]:
print("Best epoch:", result_exp6["best_epoch"])
print(
    "Best dev balanced accuracy:",
    result_exp6["best_dev_balanced_accuracy"],
)

display(
    result_exp6["history"][
        [
            "epoch",
            "train_loss",
            "dev_loss",
            "train_accuracy",
            "dev_accuracy",
            "train_balanced_accuracy",
            "dev_balanced_accuracy",
            "train_f1",
            "dev_f1",
            "train_roc_auc",
            "dev_roc_auc",
        ]
    ]
)

Best epoch: 6
Best dev balanced accuracy: 0.6471861471861472


,epoch,train_loss,dev_loss,train_accuracy,dev_accuracy,train_balanced_accuracy,dev_balanced_accuracy,train_f1,dev_f1,train_roc_auc,dev_roc_auc
0,1,0.989916,1.116071,0.427184,0.65625,0.538444,0.500000,0.438095,0.000000,0.584809,0.502165
1,2,1.002416,1.053665,0.728155,0.31250,0.580149,0.432900,0.333333,0.450000,0.523299,0.580087
2,3,1.002066,1.055235,0.582524,0.50000,0.541705,0.575758,0.376812,0.529412,0.532153,0.588745
3,4,1.009389,1.086244,0.368932,0.62500,0.550326,0.476190,0.462810,0.000000,0.606710,0.575758
4,5,0.977784,1.110316,0.737864,0.65625,0.534483,0.500000,0.129032,0.000000,0.678006,0.580087
5,6,1.002671,1.041877,0.407767,0.59375,0.535415,0.647186,0.440367,0.580645,0.543336,0.623377
6,7,1.005602,1.052761,0.699029,0.62500,0.507456,0.627706,0.114286,0.538462,0.596459,0.623377
7,8,0.902935,1.004524,0.524272,0.50000,0.626980,0.575758,0.505051,0.529412,0.796831,0.623377
8,9,0.790135,1.390129,0.718447,0.68750,0.730662,0.567100,0.602740,0.285714,0.812675,0.601732
9,10,0.645152,1.382989,0.796117,0.68750,0.805685,0.588745,0.695652,0.375000,0.868127,0.610390


### Comparison Table

In [82]:
results = {
    "Weighted 768": result,
    "Attention 768": result_exp2,
    "Weighted 256 PCA": result_exp3,
    "Attention 256 PCA": result_exp4,
    "Weighted 256 Linear": result_exp5,
    "Attention 256 Linear": result_exp6,
}


comparison_rows = []

for experiment_name, experiment_result in results.items():

    metrics = experiment_result["test_metrics"]

    comparison_rows.append(
        {
            "Experiment": experiment_name,
            "Best Epoch": experiment_result["best_epoch"],
            "Best Dev Bal. Acc.": experiment_result[
                "best_dev_balanced_accuracy"
            ],
            "Test Accuracy": metrics["accuracy"],
            "Test Bal. Acc.": metrics["balanced_accuracy"],
            "Test F1": metrics["f1"],
            "Test ROC-AUC": metrics["roc_auc"],
            "Test Loss": metrics["loss"],
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)

display(
    comparison_df.round(4)
)

,Experiment,Best Epoch,Best Dev Bal. Acc.,Test Accuracy,Test Bal. Acc.,Test F1,Test ROC-AUC,Test Loss
0,Weighted 768,6,0.6515,0.4348,0.3929,0.2353,0.4129,1.0441
1,Attention 768,5,0.5606,0.4565,0.4286,0.2857,0.3906,1.0798
2,Weighted 256 PCA,3,0.6082,0.5652,0.4062,0.0000,0.3750,1.0338
3,Attention 256 PCA,3,0.6082,0.5652,0.4464,0.1667,0.4330,1.0285
4,Weighted 256 Linear,7,0.6061,0.4565,0.3884,0.1935,0.3906,1.0844
5,Attention 256 Linear,6,0.6472,0.3913,0.4219,0.3333,0.5112,1.0262


In [83]:
comparison_df["Predicted 0"] = [
    (
        r["test_predictions"]["predicted_label"] == 0
    ).sum()
    for r in results.values()
]

comparison_df["Predicted 1"] = [
    (
        r["test_predictions"]["predicted_label"] == 1
    ).sum()
    for r in results.values()
]

display(
    comparison_df.round(4)
)

,Experiment,Best Epoch,Best Dev Bal. Acc.,Test Accuracy,Test Bal. Acc.,Test F1,Test ROC-AUC,Test Loss,Predicted 0,Predicted 1
0,Weighted 768,6,0.6515,0.4348,0.3929,0.2353,0.4129,1.0441,26,20
1,Attention 768,5,0.5606,0.4565,0.4286,0.2857,0.3906,1.0798,25,21
2,Weighted 256 PCA,3,0.6082,0.5652,0.4062,0.0000,0.3750,1.0338,40,6
3,Attention 256 PCA,3,0.6082,0.5652,0.4464,0.1667,0.4330,1.0285,36,10
4,Weighted 256 Linear,7,0.6061,0.4565,0.3884,0.1935,0.3906,1.0844,29,17
5,Attention 256 Linear,6,0.6472,0.3913,0.4219,0.3333,0.5112,1.0262,18,28


## 14. Inspect attention weights

For attention experiments, the model returns audio/text attention weights. Because chunk order and metadata were saved, these can later be mapped back to:
- audio `start_seconds` / `end_seconds`;
- text `token_start` / `token_end`.

High attention means the learned pooling module gave that chunk more weight. It should not automatically be interpreted as a causal or clinical explanation.

In [ ]:
@torch.inference_mode()
def get_attention_for_participant(
    model,
    participant_id,
    audio_pca=None,
    text_pca=None,
):
    dataset = MultimodalChunkDataset(
        [participant_id],
        audio_pca=audio_pca,
        text_pca=text_pca,
    )

    batch = multimodal_collate(
        [dataset[0]]
    )
    batch = move_batch(batch)

    model.eval()

    out = model(
        batch["audio_embeddings"],
        batch["audio_lengths"],
        batch["text_embeddings"],
        batch["text_lengths"],
    )

    n_audio = int(
        (batch["audio_lengths"][0] > 0).sum().item()
    )
    n_text = int(
        (batch["text_lengths"][0] > 0).sum().item()
    )

    return {
        "audio_attention": (
            out["audio_weights"][0, :n_audio]
            .cpu()
            .numpy()
        ),
        "text_attention": (
            out["text_weights"][0, :n_text]
            .cpu()
            .numpy()
        ),
    }

## 15. Validation checks

Run these before full experiments.

In [ ]:
def validate_saved_participant(participant_id):
    audio, text = load_participant_modalities(participant_id)

    assert audio["embeddings"].shape[1] == 768
    assert text["embeddings"].shape[1] == 768

    assert len(audio["embeddings"]) == len(audio["lengths"])
    assert len(text["embeddings"]) == len(text["lengths"])

    assert np.all(audio["lengths"] > 0)
    assert np.all(text["lengths"] > 0)

    assert np.isfinite(audio["embeddings"]).all()
    assert np.isfinite(text["embeddings"]).all()

    assert audio["phq8_binary"] == text["phq8_binary"]
    assert audio["dataset"] == text["dataset"]

    return {
        "Participant_ID": str(participant_id),
        "audio_shape": audio["embeddings"].shape,
        "text_shape": text["embeddings"].shape,
        "PHQ8_Binary": audio["phq8_binary"],
        "Dataset": audio["dataset"],
    }


# Example after extraction:
# display(validate_saved_participant(train_ids[0]))